**Generate Graphs + features**

In [1]:
import numpy as np
import networkx as nx
import torch
from torch_geometric.utils import from_networkx
from torch_geometric.data import data

In [19]:
def generate_sbm(n = 500, k =2, p_in = 0.8, p_out = 0.1):
  sizes = [n//k] * k

  # probability matrix 
  probs = [[p_in if i == j else p_out for j in range(k)] for i in range(k)]
  G = nx.stochastic_block_model(sizes, probs)

  labels = []
  for i, size in enumerate(sizes):
    labels.extend([i]* size)
  return G, np.array(labels)

# def generate_features(labels, dim = 16):
#   features = []
#   for label in labels:
#     center = np.zeros(dim)
#     center[label] = 1
#     features.append(center + 0.5 * np.random.randn(dim))
#   return np.array(features)

def generate_features(labels, dim=16):
    return np.random.randn(len(labels), dim)

**Convert to pyG format**

In [3]:
def create_pyg_data(G, labels, features):
  data = from_networkx(G)

  data.x = torch.tensor(features, dtype = torch.float)
  data.y = torch.tensor(labels, dtype = torch.long)

  # train/test split
  n = len(labels)
  idx = torch.randperm(n)

  train_mask = torch.zeros(n, dtype = torch.bool)
  test_mask = torch.zeros(n, dtype=torch.bool)

  train_mask[idx[:int(0.6*n)]]  = True
  test_mask[idx[int(0.6*n):]] = True

  data.train_mask = train_mask
  data.test_mask = test_mask

  return data

**GCN model**

In [4]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric .nn import GCNConv

In [20]:
class GCN(nn.Module):
  def __init__(self, in_dim, hidden_dim, out_dim):
    super().__init__()
    self.conv1 = GCNConv(in_dim, hidden_dim, add_self_loops= False)
    self.conv2 = GCNConv(hidden_dim, out_dim, add_self_loops= False)
  def forward(self, x, edge_index):
    x = self.conv1(x, edge_index)
    x = F.relu(x)
    x = self.conv2(x, edge_index)
    return x

In [21]:
def train(data, model, optimizer):
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()

    return loss.item()

def test(data, model):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
    acc = int(correct) / int(data.test_mask.sum())
    return acc

In [33]:
def run_experiment(model, p_in, p_out):
    G, labels = generate_sbm(p_in=p_in, p_out=p_out)
    n = len(labels)
    # features = generate_features(labels)
    features = np.random.randn(n, 16)

    data_ = create_pyg_data(G, labels, features)

    # model = GCN(in_dim=16, hidden_dim=32, out_dim=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(100):
        loss = train(data_, model, optimizer)

    acc = test(data_, model)
    return acc

In [12]:
G, labels = generate_sbm(p_in = 0.1, p_out= 0.8)
features = generate_features(labels)

def edge_homophily(G, labels):
    same = sum(labels[u] == labels[v] for u, v in G.edges())
    return same / len(G.edges())

print(edge_homophily(G, labels))

0.11292837651061417


In [34]:
def generate_features(labels, dim=16):
    return np.random.randn(len(labels), dim)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.lin1 = nn.Linear(in_dim, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x, edge_index=None):
        x = F.relu(self.lin1(x))
        return self.lin2(x)

In [35]:
model_GCN = GCN(in_dim=16, hidden_dim=32, out_dim=2)
model_MLP = MLP(in_dim=16, hidden_dim=32, out_dim=2)

acc_homo_GCN = run_experiment(model_GCN, p_in = 0.9, p_out = 0.05)
acc_homo_MLP = run_experiment(model_MLP, p_in = 0.9, p_out = 0.05)

acc_hetero_GCN = run_experiment(model_GCN, p_in = 0.9, p_out = 0.05)
acc_hetero_MLP = run_experiment(model_MLP,p_in=0.05, p_out=0.9)

print("Homophilic Accuracy GCN:", acc_homo_GCN)
print("Homophilic Accuracy MLP:", acc_homo_MLP)
print("Heterophilic Accuracy GCN:", acc_hetero_GCN)
print("Heterophilic Accuracy MLP:", acc_hetero_MLP)

Homophilic Accuracy GCN: 1.0
Homophilic Accuracy MLP: 0.46
Heterophilic Accuracy GCN: 1.0
Heterophilic Accuracy MLP: 0.495


In [ ]:
# Homophilic
acc_homo = run_experiment(p_in=0.9, p_out=0.05)

# Heterophilic
acc_hetero = run_experiment(p_in=0.05, p_out=0.9)

print("Homophilic Accuracy:", acc_homo)
print("Heterophilic Accuracy:", acc_hetero)

Homophilic Accuracy: 1.0
Heterophilic Accuracy: 1.0


In [30]:
print(data_homo.train_mask.sum(), data_homo.test_mask.sum())
print((data_homo.train_mask & data_homo.test_mask).sum())

tensor(300) tensor(200)
tensor(0)


In [16]:
# Extreme Homophilic
G_homo, y_homo = generate_sbm(p_in=0.9, p_out=0.05)

# Extreme Heterophilic
G_hetero, y_hetero = generate_sbm(p_in=0.05, p_out=0.9)

print("Homo h:", edge_homophily(G_homo, y_homo))
print("Hetero h:", edge_homophily(G_hetero, y_hetero))

Homo h: 0.9463647400013488
Hetero h: 0.052982361653426686
